# Vietnamese Sentiment Analysis on Student Feedback

Baselines:

- TF-IDF + Logistic Regression
- TF-IDF + Linear SVM

Dataset: `uitnlp/vietnamese_students_feedback`

## 1. Install and Import Libraries

Run this notebook on Kaggle with Internet turned on.

In [ ]:
!pip install -q datasets gdown

In [ ]:
import unicodedata
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import gdown

from datasets import load_dataset
from sklearn.base import clone
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix,
    f1_score,
    precision_recall_fscore_support,
)
from sklearn.pipeline import FeatureUnion, Pipeline
from sklearn.svm import LinearSVC

RANDOM_STATE = 42
DATASET_NAME = "uitnlp/vietnamese_students_feedback"
TEXT_COL = "sentence"
LABEL_COL = "sentiment"
OUTPUT_DIR = Path("outputs")
CACHE_DIR = Path("vsfc_data")
PROCESSED_DATA_DIR = Path("Sentiment-Analysis-Vietnamese") / "dataset" / "processed"
MODEL_TEXT_COL = "processed_ml"
PROCESSED_FILENAMES = {
    "train": "train_processed.csv",
    "validation": "valid_processed.csv",
    "test": "test_processed.csv",
}
OUTPUT_DIR.mkdir(exist_ok=True)
CACHE_DIR.mkdir(exist_ok=True)

LABEL_NAMES = {
    0: "negative",
    1: "neutral",
    2: "positive",
}

DATA_FILE_IDS = {
    "train": {
        "sentences": "1nzak5OkrheRV1ltOGCXkT671bmjODLhP",
        "sentiments": "1ye-gOZIBqXdKOoi_YxvpT6FeRNmViPPv",
        "topics": "14MuDtwMnNOcr4z_8KdpxprjbwaQ7lJ_C",
    },
    "validation": {
        "sentences": "1sMJSR3oRfPc3fe1gK-V3W5F24tov_517",
        "sentiments": "1GiY1AOp41dLXIIkgES4422AuDwmbUseL",
        "topics": "1DwLgDEaFWQe8mOd7EpF-xqMEbDLfdT-W",
    },
    "test": {
        "sentences": "1aNMOeZZbNwSRkjyCWAGtNCMa3YrshR-n",
        "sentiments": "1vkQS5gI0is4ACU58-AbWusnemw7KZNfO",
        "topics": "1_ArMpDguVsbUGl-xSMkTF_p5KpZrmpSB",
    },
}


def read_lines(path):
    return Path(path).read_text(encoding="utf-8").splitlines()


def download_drive_file(file_id, output_path):
    output_path = Path(output_path)
    if output_path.exists() and output_path.stat().st_size > 0:
        return output_path
    output_path.parent.mkdir(parents=True, exist_ok=True)
    url = f"https://drive.google.com/uc?id={file_id}"
    gdown.download(url=url, output=str(output_path), quiet=False, fuzzy=True)
    if not output_path.exists() or output_path.stat().st_size == 0:
        raise RuntimeError(f"Could not download {output_path.name} from Google Drive.")
    return output_path


def load_split_from_drive(split_name):
    paths = {}
    for field_name, file_id in DATA_FILE_IDS[split_name].items():
        paths[field_name] = download_drive_file(file_id, CACHE_DIR / split_name / f"{field_name}.txt")

    sentences = read_lines(paths["sentences"])
    sentiments = [int(value.strip()) for value in read_lines(paths["sentiments"])]
    topics = [int(value.strip()) for value in read_lines(paths["topics"])]

    if not (len(sentences) == len(sentiments) == len(topics)):
        raise ValueError(
            f"Length mismatch in {split_name}: "
            f"{len(sentences)} sentences, {len(sentiments)} sentiments, {len(topics)} topics"
        )

    return pd.DataFrame(
        {
            "sentence": sentences,
            "sentiment": sentiments,
            "topic": topics,
        }
    )


def load_vsfc_dataset():
    try:
        return load_dataset(DATASET_NAME)
    except RuntimeError as exc:
        if "Dataset scripts are no longer supported" not in str(exc):
            raise
        print("Hugging Face datasets no longer supports this old dataset script.")
        print("Falling back to the original Google Drive files referenced by the dataset script.")
        return {
            "train": load_split_from_drive("train"),
            "validation": load_split_from_drive("validation"),
            "test": load_split_from_drive("test"),
        }


## 2. Load Dataset

This notebook first tries to use the already preprocessed files from `Sentiment-Analysis-Vietnamese/dataset/processed`.
If those files are not available on Kaggle, it falls back to loading the original dataset.

In [ ]:
def find_processed_data_dir():
    required = list(PROCESSED_FILENAMES.values())
    candidate_dirs = [
        PROCESSED_DATA_DIR,
        Path("."),
        Path("dataset") / "processed",
        Path("/kaggle/working"),
    ]

    for directory in candidate_dirs:
        if directory.exists() and all((directory / filename).exists() for filename in required):
            return directory

    kaggle_input = Path("/kaggle/input")
    if kaggle_input.exists():
        for train_file in kaggle_input.rglob(PROCESSED_FILENAMES["train"]):
            directory = train_file.parent
            if all((directory / filename).exists() for filename in required):
                return directory

    return None


def load_processed_split(split_name, processed_dir):
    path = processed_dir / PROCESSED_FILENAMES[split_name]
    df = pd.read_csv(path)
    if MODEL_TEXT_COL not in df.columns:
        raise ValueError(f"{path} does not contain required column: {MODEL_TEXT_COL}")
    df = df[[TEXT_COL, LABEL_COL, MODEL_TEXT_COL]].copy()
    df[TEXT_COL] = df[TEXT_COL].map(normalize_vietnamese_text)
    df[MODEL_TEXT_COL] = df[MODEL_TEXT_COL].fillna("").map(normalize_vietnamese_text)
    df[LABEL_COL] = df[LABEL_COL].astype(int)
    return df


def load_dataframes():
    processed_dir = find_processed_data_dir()
    if processed_dir is not None:
        print(f"Using preprocessed CSV files from: {processed_dir}")
        return (
            load_processed_split("train", processed_dir),
            load_processed_split("validation", processed_dir),
            load_processed_split("test", processed_dir),
            MODEL_TEXT_COL,
        )

    print("Preprocessed CSV files were not found.")
    print("Falling back to light text normalization from the original dataset.")
    ds = load_vsfc_dataset()
    return (
        dataset_split_to_frame(ds["train"]),
        dataset_split_to_frame(ds["validation"]),
        dataset_split_to_frame(ds["test"]),
        TEXT_COL,
    )


In [ ]:
# Dataframes are loaded in the next section after preprocessing helpers are defined.

## 3. Prepare DataFrames

For TF-IDF baselines, the preferred input column is `processed_ml`, because it has already been normalized and word-segmented.
If preprocessed files are unavailable, the notebook uses the original `sentence` column with light normalization.

In [ ]:
def normalize_vietnamese_text(text):
    if text is None:
        return ""
    text = str(text)
    text = unicodedata.normalize("NFC", text)
    text = text.lower()
    text = " ".join(text.split())
    return text


def dataset_split_to_frame(split):
    df = pd.DataFrame(split)
    df = df[[TEXT_COL, LABEL_COL]].copy()
    df[TEXT_COL] = df[TEXT_COL].map(normalize_vietnamese_text)
    df[LABEL_COL] = df[LABEL_COL].astype(int)
    return df


train_df, valid_df, test_df, model_text_col = load_dataframes()

print("Train:", train_df.shape)
print("Validation:", valid_df.shape)
print("Test:", test_df.shape)
print("Text column used for TF-IDF:", model_text_col)

train_df.head()

## 4. Exploratory Data Analysis

In [ ]:
for name, df in [("train", train_df), ("validation", valid_df), ("test", test_df)]:
    counts = df[LABEL_COL].value_counts().sort_index()
    print(f"\n{name} label distribution:")
    for label_id, count in counts.items():
        pct = count / len(df) * 100
        print(f"  {label_id} ({LABEL_NAMES[label_id]:>8}): {count:>5} ({pct:5.2f}%)")

In [ ]:
plt.figure(figsize=(7, 4))
sns.countplot(data=train_df, x=LABEL_COL, order=[0, 1, 2])
plt.xticks([0, 1, 2], [LABEL_NAMES[i] for i in [0, 1, 2]])
plt.title("Train Label Distribution")
plt.xlabel("Sentiment")
plt.ylabel("Count")
plt.tight_layout()
plt.show()

In [ ]:
train_df["num_words"] = train_df[TEXT_COL].str.split().map(len)

plt.figure(figsize=(8, 4))
sns.histplot(train_df["num_words"], bins=40, kde=True)
plt.title("Sentence Length Distribution - Train Set")
plt.xlabel("Number of words")
plt.ylabel("Count")
plt.tight_layout()
plt.show()

train_df["num_words"].describe()

## 5. Build TF-IDF Features

We combine:

- Word-level TF-IDF: captures meaningful words and phrases.
- Character-level TF-IDF: helps Vietnamese text because small spelling variations and syllable patterns still carry signal.

In [ ]:
def build_tfidf_features():
    word_tfidf = TfidfVectorizer(
        analyzer="word",
        ngram_range=(1, 2),
        min_df=2,
        max_df=0.95,
        max_features=50_000,
        sublinear_tf=True,
    )

    char_tfidf = TfidfVectorizer(
        analyzer="char_wb",
        ngram_range=(3, 5),
        min_df=2,
        max_df=0.95,
        max_features=50_000,
        sublinear_tf=True,
    )

    return FeatureUnion(
        transformer_list=[
            ("word_tfidf", word_tfidf),
            ("char_tfidf", char_tfidf),
        ],
        n_jobs=-1,
    )

## 6. Train Baseline Models

In [ ]:
train_full_df = pd.concat([train_df, valid_df], ignore_index=True)

X_train = train_full_df[model_text_col]
y_train = train_full_df[LABEL_COL]
X_test = test_df[model_text_col]
y_test = test_df[LABEL_COL]

tfidf_features = build_tfidf_features()

models = {
    "tfidf_logistic_regression": Pipeline(
        steps=[
            ("tfidf", clone(tfidf_features)),
            (
                "clf",
                LogisticRegression(
                    C=2.0,
                    class_weight="balanced",
                    max_iter=2000,
                    solver="saga",
                    n_jobs=-1,
                    random_state=RANDOM_STATE,
                ),
            ),
        ]
    ),
    "tfidf_linear_svm": Pipeline(
        steps=[
            ("tfidf", clone(tfidf_features)),
            (
                "clf",
                LinearSVC(
                    C=1.0,
                    class_weight="balanced",
                    random_state=RANDOM_STATE,
                ),
            ),
        ]
    ),
}

for model_name, model in models.items():
    print(f"Training {model_name}...")
    model.fit(X_train, y_train)
    print("Done")

## 7. Evaluate Models

In [ ]:
def evaluate_model(model_name, model, X_test, y_test):
    y_pred = model.predict(X_test)

    accuracy = accuracy_score(y_test, y_pred)
    macro_f1 = f1_score(y_test, y_pred, average="macro")
    weighted_f1 = f1_score(y_test, y_pred, average="weighted")
    macro_precision, macro_recall, _, _ = precision_recall_fscore_support(
        y_test,
        y_pred,
        average="macro",
        zero_division=0,
    )

    print(f"\n=== {model_name} ===")
    print(f"Accuracy:        {accuracy:.4f}")
    print(f"Macro Precision: {macro_precision:.4f}")
    print(f"Macro Recall:    {macro_recall:.4f}")
    print(f"Macro F1:        {macro_f1:.4f}")
    print(f"Weighted F1:     {weighted_f1:.4f}")
    print("\nClassification report:")
    print(
        classification_report(
            y_test,
            y_pred,
            labels=[0, 1, 2],
            target_names=[LABEL_NAMES[i] for i in [0, 1, 2]],
            digits=4,
            zero_division=0,
        )
    )

    cm = confusion_matrix(y_test, y_pred, labels=[0, 1, 2])
    plt.figure(figsize=(6, 5))
    sns.heatmap(
        cm,
        annot=True,
        fmt="d",
        cmap="Blues",
        xticklabels=[LABEL_NAMES[i] for i in [0, 1, 2]],
        yticklabels=[LABEL_NAMES[i] for i in [0, 1, 2]],
    )
    plt.title(f"Confusion Matrix - {model_name}")
    plt.xlabel("Predicted label")
    plt.ylabel("True label")
    plt.tight_layout()
    plt.savefig(OUTPUT_DIR / f"{model_name}_confusion_matrix.png", dpi=200)
    plt.show()

    return {
        "model": model_name,
        "accuracy": accuracy,
        "macro_precision": macro_precision,
        "macro_recall": macro_recall,
        "macro_f1": macro_f1,
        "weighted_f1": weighted_f1,
    }, y_pred

In [ ]:
results = []
predictions = {}

for model_name, model in models.items():
    result, y_pred = evaluate_model(model_name, model, X_test, y_test)
    results.append(result)
    predictions[model_name] = y_pred

results_df = pd.DataFrame(results).sort_values("macro_f1", ascending=False)
results_df

## 8. Save Results and Predictions

In [ ]:
results_df.to_csv(OUTPUT_DIR / "baseline_results.csv", index=False, encoding="utf-8-sig")

for model_name, y_pred in predictions.items():
    output = test_df[[TEXT_COL, LABEL_COL]].copy()
    output["predicted_sentiment"] = y_pred
    output["true_label_name"] = output[LABEL_COL].map(LABEL_NAMES)
    output["predicted_label_name"] = output["predicted_sentiment"].map(LABEL_NAMES)
    output["is_correct"] = output[LABEL_COL] == output["predicted_sentiment"]
    output.to_csv(OUTPUT_DIR / f"{model_name}_test_predictions.csv", index=False, encoding="utf-8-sig")

print(f"Saved outputs to: {OUTPUT_DIR.resolve()}")

## 9. Error Analysis Examples

Use these examples for the report section about misclassified cases.

In [ ]:
best_model_name = results_df.iloc[0]["model"]
print("Best baseline:", best_model_name)

analysis_df = test_df[[TEXT_COL, LABEL_COL]].copy()
analysis_df["predicted_sentiment"] = predictions[best_model_name]
analysis_df["true_label_name"] = analysis_df[LABEL_COL].map(LABEL_NAMES)
analysis_df["predicted_label_name"] = analysis_df["predicted_sentiment"].map(LABEL_NAMES)
analysis_df["is_correct"] = analysis_df[LABEL_COL] == analysis_df["predicted_sentiment"]

analysis_df[analysis_df["is_correct"] == False].head(20)